Copyright © 2025, Andrea Mastropietro. All rights reserved.

This code is licensed under the MIT License.

See the LICENSE file in the project root for more information.

## Analysis of anchor and non-anchor atoms using a control based on E(3)-transformations (distance-based vs Coulomb-matrix based)

In [1]:
import os
# os.environ["http_proxy"] = "http://web-proxy.informatik.uni-bonn.de:3128"
# os.environ["https_proxy"] = "http://web-proxy.informatik.uni-bonn.de:3128"

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
# Standard library imports
import copy
import os

# Third-party imports
import yaml
import torch
import numpy as np

# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

# Project-specific imports
from src.difflinker.datasets import get_dataloader
from src.difflinker.lightning import DDPM

In [3]:
with open('../config.yml', 'r') as file:
    config = yaml.safe_load(file)

checkpoint = "../" + config['CHECKPOINT']
DATA_FOLDER = "../" + config['DATA_FOLDER']
DATASET_NAME = config['DATASET_NAME']
device = config['DEVICE'] if torch.cuda.is_available() else 'cpu'
NUM_SAMPLES = config['NUM_SAMPLES']
P = config['P']
ATOM_TYPE_PERTURBATION = config['ATOM_TYPE_PERTURBATION']

SAVE_PLOT_FOLDER = "../results/plots/frequency_analysis/"
SHAPLEY_VALUE_FOLDER = f"../results/explanations/{DATASET_NAME}/" 
POSITIONS_FOLDER = f"../results/explanations/{DATASET_NAME}/" 

if ATOM_TYPE_PERTURBATION:
    SAVE_PLOT_FOLDER += "including_atom_type_perturbation/"
    
os.makedirs(SAVE_PLOT_FOLDER, exist_ok=True)

Load model to load the data

In [4]:
model = DDPM.load_from_checkpoint(checkpoint, map_location=device)
model.val_data_prefix = DATASET_NAME

print(f"Running device: {device}")

model.data_path = DATA_FOLDER

model = model.eval().to(device)
model.setup(stage='val')
dataloader = get_dataloader(
    model.val_dataset,
    batch_size=1
)

c:\Users\Mastro\anaconda3\envs\diff_explainer\lib\site-packages\lightning_fabric\utilities\cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
Lightning automatically upgraded your

Running device: cuda:0


Load data samples for fragment, linker and anchor indices

In [5]:
data_list = []
sampled = 0
data_dict = {}
for data in dataloader:
    if sampled < NUM_SAMPLES:
        data_list.append(data)
        sampled += 1

#print all fragment masks, linker masks, and anchor masks
for i in range(len(data_list)):
    data_dict[i] = {}
    data_dict[i]["fragment_mask"] = data_list[i]['fragment_mask'].squeeze(0)
    data_dict[i]["linker_mask"] = data_list[i]['linker_mask'].squeeze(0)
    data_dict[i]["anchors"] = data_list[i]['anchors'].squeeze(0)


### Analyzing distance-based Shapley values

Read Shapley values from disk

In [6]:
seed_list = [42]
# STRATEGY = "hausdorff_distance"
for seed in seed_list:
    for i in range(NUM_SAMPLES):
        with open(f"{SHAPLEY_VALUE_FOLDER}explanations_seed_{str(seed)}/shapley_values/shapley_values_atoms_{i}.txt", "r") as f:
            f.readline()
            f.readline()
            shapley_values = []
            for line in f:
                if line == "\n":
                    break
                row = line.strip().split(",")
                shapley_values.append(float(row[1]))
            dict_key_name = f"shapley_values_{seed}"
            data_dict[i][dict_key_name] = shapley_values


read final positions from disk

In [7]:
for seed in seed_list:
    for i in range(NUM_SAMPLES):
        with open(f"{POSITIONS_FOLDER}explanations_seed_{str(seed)}/mapping/graphs/{i}/{i}_0_.xyz", "r") as f: #file with final atom positions
            num_atoms = int(f.readline().strip())
            f.readline()
            positions = []*num_atoms
            for line in f:
                row = line.strip().split(" ")[1:]
                coords = [float(x) for x in row]
                positions.append(coords)
            dict_key_name = f"positions_{seed}"
            data_dict[i][dict_key_name] = positions

Compute center of mass of linker atoms

In [8]:
for i in range(len(data_list)):
    linker_mask = data_dict[i]['linker_mask'].bool().cpu().numpy()
    for seed in seed_list:
        dict_key_name = f"positions_{seed}"
        positions = np.array(data_dict[i][dict_key_name])
        linker_positions = positions[linker_mask.squeeze()]
        com_linker = np.mean(linker_positions, axis=0)
        dict_key_name = f"com_linker_{seed}"
        data_dict[i][dict_key_name] = com_linker

Compute distance between fragment atoms and linker COM

In [9]:
for i in range(len(data_list)):
    fragment_mask = data_dict[i]['fragment_mask'].bool().cpu().numpy()
    for seed in seed_list:
        fragment_positions = np.array(data_dict[i][f'positions_{seed}'])[fragment_mask.squeeze()]
        com_linker = data_dict[i][f'com_linker_{seed}']
        distances = np.linalg.norm(fragment_positions - com_linker, axis=1)
        dict_key_name = f'distances_{seed}'
        data_dict[i][dict_key_name] = distances

Boxplot generation

In [10]:
for i in range(len(data_list)):
    for seed in seed_list:
        dict_key_name = f"shapley_values_{seed}"
        shapley_values = -np.array(data_dict[i][dict_key_name])
        min_val = np.min(shapley_values)
        max_val = np.max(shapley_values)
        normalized_shapley_values = 2 * (shapley_values - min_val) / (max_val - min_val) - 1
        dict_key_name = f"normalized_shapley_values_{seed}"
        data_dict[i][dict_key_name] = normalized_shapley_values.tolist()

In [11]:
for i in range(len(data_list)):
    for seed in seed_list:
        dict_key_name = f'distances_{seed}'
        distances = np.array(data_dict[i][dict_key_name])
        min_dist = np.min(distances)
        max_dist = np.max(distances)
        normalized_distances = (distances - min_dist) / (max_dist - min_dist)
        dict_key_name = f'normalized_distances_{seed}'
        data_dict[i][dict_key_name] = normalized_distances.tolist()

In [15]:
# Frequency of anchor atoms appearing in top-5 normalized Shapley atoms (only seed=42)
top_k = 5
seed = 42
anchor_topk_stats = {}

total_hits = 0
total_topk_slots = 0
total_anchor_atoms = 0
samples_with_anchor_in_topk = 0
valid_samples = 0

for i in range(len(data_list)):
    shapley = np.array(data_dict[i][f"normalized_shapley_values_{seed}"])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]

    # Robust anchor extraction (supports mask-style or index-style anchors)
    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    k = min(top_k, shapley.size)
    topk_idx = np.argsort(shapley)[::-1][:k]

    # If shapley is fragment-only, map anchors to fragment-local indices
    if shapley.size == int(fragment_mask.sum()):
        fragment_global_idx = np.where(fragment_mask)[0]
        global_to_local = {g: l for l, g in enumerate(fragment_global_idx)}
        anchor_idx_for_comparison = np.array(
            [global_to_local[g] for g in anchor_global_idx if g in global_to_local], dtype=int
        )
    else:
        # Assume shapley is over all atoms
        anchor_idx_for_comparison = anchor_global_idx.astype(int)

    hits = np.intersect1d(topk_idx, anchor_idx_for_comparison).size

    total_hits += hits
    total_topk_slots += k
    total_anchor_atoms += anchor_idx_for_comparison.size
    samples_with_anchor_in_topk += int(hits > 0)
    valid_samples += 1

anchor_topk_stats[seed] = {
    "hit_rate_in_topk_slots": (total_hits / total_topk_slots) if total_topk_slots > 0 else np.nan,
    "anchor_coverage_rate": (total_hits / total_anchor_atoms) if total_anchor_atoms > 0 else np.nan,
    "sample_frequency_any_anchor_in_topk": (samples_with_anchor_in_topk / valid_samples) if valid_samples > 0 else np.nan,
    "total_hits": total_hits,
    "total_topk_slots": total_topk_slots,
    "total_anchor_atoms": total_anchor_atoms,
    "valid_samples": valid_samples,
}

stats = anchor_topk_stats[seed]
print(f"Seed {seed}:")
print(f"  hit_rate_in_topk_slots            = {stats['hit_rate_in_topk_slots']:.4f}")
print(f"  anchor_coverage_rate              = {stats['anchor_coverage_rate']:.4f}")
print(f"  sample_frequency_any_anchor_in_topk = {stats['sample_frequency_any_anchor_in_topk']:.4f}")
print(f"  (hits={stats['total_hits']}, topk_slots={stats['total_topk_slots']}, "
      f"anchors={stats['total_anchor_atoms']}, samples={stats['valid_samples']})")

Seed 42:
  hit_rate_in_topk_slots            = 0.2200
  anchor_coverage_rate              = 0.5500
  sample_frequency_any_anchor_in_topk = 0.7667
  (hits=33, topk_slots=150, anchors=60, samples=30)


In [19]:
# Frequency analysis for anchor atoms and their fragment-neighbors in top-k Shapley atoms
seed = 42 if "seed" not in globals() else seed
top_k = 5 if "top_k" not in globals() else top_k
neighbor_k = 2  # number of nearest fragment neighbors per anchor

anchor_neighbor_topk_stats = {}

tot_anchor_hits = tot_neighbor_hits = tot_combined_hits = 0
tot_topk_slots = 0
tot_anchor_atoms = tot_neighbor_atoms = tot_combined_atoms = 0
samples_anchor_any = samples_neighbor_any = samples_combined_any = 0
valid_samples = 0

for i in range(len(data_list)):
    shapley = np.array(data_dict[i][f"normalized_shapley_values_{seed}"])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]
    fragment_global_idx = np.where(fragment_mask)[0]

    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    # anchor extraction (mask-style or index-style)
    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    # keep only anchors inside fragment atoms
    anchor_global_idx = np.array([a for a in anchor_global_idx if a in set(fragment_global_idx)], dtype=int)

    # top-k indices in shapley space
    k = min(top_k, shapley.size)
    topk_idx = np.argsort(shapley)[::-1][:k]

    # map global atom index -> shapley index space
    if shapley.size == fragment_global_idx.size:
        global_to_shapley = {g: l for l, g in enumerate(fragment_global_idx)}
    else:
        global_to_shapley = {g: g for g in range(min(n_atoms, shapley.size))}

    anchor_idx = np.array([global_to_shapley[g] for g in anchor_global_idx if g in global_to_shapley], dtype=int)

    # nearest fragment neighbors of anchors from coordinates
    positions_key = f"positions_{seed}"
    if positions_key not in data_dict[i]:
        continue
    pos = np.array(data_dict[i][positions_key])
    frag_pos = pos[fragment_global_idx]

    frag_global_to_local = {g: l for l, g in enumerate(fragment_global_idx)}
    anchor_local = np.array([frag_global_to_local[g] for g in anchor_global_idx if g in frag_global_to_local], dtype=int)

    neighbor_global_set = set()
    if anchor_local.size > 0:
        for a_local in anchor_local:
            d = np.linalg.norm(frag_pos - frag_pos[a_local], axis=1)
            order = np.argsort(d)
            # skip itself and pick nearest non-anchor fragment atoms
            picked = 0
            for idx_local in order:
                g_idx = fragment_global_idx[idx_local]
                if g_idx == fragment_global_idx[a_local] or g_idx in set(anchor_global_idx):
                    continue
                neighbor_global_set.add(int(g_idx))
                picked += 1
                if picked >= neighbor_k:
                    break

    neighbor_idx = np.array(
        [global_to_shapley[g] for g in sorted(neighbor_global_set) if g in global_to_shapley],
        dtype=int
    )

    combined_idx = np.unique(np.concatenate([anchor_idx, neighbor_idx])) if (anchor_idx.size + neighbor_idx.size) > 0 else np.array([], dtype=int)

    anchor_hits = np.intersect1d(topk_idx, anchor_idx).size
    neighbor_hits = np.intersect1d(topk_idx, neighbor_idx).size
    combined_hits = np.intersect1d(topk_idx, combined_idx).size

    tot_anchor_hits += anchor_hits
    tot_neighbor_hits += neighbor_hits
    tot_combined_hits += combined_hits
    tot_topk_slots += k

    tot_anchor_atoms += anchor_idx.size
    tot_neighbor_atoms += neighbor_idx.size
    tot_combined_atoms += combined_idx.size

    samples_anchor_any += int(anchor_hits > 0)
    samples_neighbor_any += int(neighbor_hits > 0)
    samples_combined_any += int(combined_hits > 0)
    valid_samples += 1

anchor_neighbor_topk_stats[seed] = {
    "anchor_hit_rate_in_topk_slots": (tot_anchor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "neighbor_hit_rate_in_topk_slots": (tot_neighbor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_or_neighbor_hit_rate_in_topk_slots": (tot_combined_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_coverage_rate": (tot_anchor_hits / tot_anchor_atoms) if tot_anchor_atoms > 0 else np.nan,
    "neighbor_coverage_rate": (tot_neighbor_hits / tot_neighbor_atoms) if tot_neighbor_atoms > 0 else np.nan,
    "anchor_or_neighbor_coverage_rate": (tot_combined_hits / tot_combined_atoms) if tot_combined_atoms > 0 else np.nan,
    "sample_freq_any_anchor_in_topk": (samples_anchor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_neighbor_in_topk": (samples_neighbor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_anchor_or_neighbor_in_topk": (samples_combined_any / valid_samples) if valid_samples > 0 else np.nan,
    "valid_samples": valid_samples,
    "neighbor_k": neighbor_k,
    "top_k": top_k,
}

stats_an = anchor_neighbor_topk_stats[seed]
print(f"Seed {seed} | top_k={top_k}, neighbor_k={neighbor_k}")
for k, v in stats_an.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Seed 42 | top_k=5, neighbor_k=2
  anchor_hit_rate_in_topk_slots: 0.2200
  neighbor_hit_rate_in_topk_slots: 0.2933
  anchor_or_neighbor_hit_rate_in_topk_slots: 0.5133
  anchor_coverage_rate: 0.5500
  neighbor_coverage_rate: 0.3667
  anchor_or_neighbor_coverage_rate: 0.4278
  sample_freq_any_anchor_in_topk: 0.7667
  sample_freq_any_neighbor_in_topk: 0.8667
  sample_freq_any_anchor_or_neighbor_in_topk: 0.9667
  valid_samples: 30
  neighbor_k: 2
  top_k: 5


In [20]:
# Frequency analysis for anchor atoms and neighbors near linker COM (distance <= 2.0 Å)
seed = 42 if "seed" not in globals() else seed
top_k = 5 if "top_k" not in globals() else top_k
distance_threshold = 2.0

anchor_com_neighbor_topk_stats = {}

tot_anchor_hits = tot_neighbor_hits = tot_combined_hits = 0
tot_topk_slots = 0
tot_anchor_atoms = tot_neighbor_atoms = tot_combined_atoms = 0
samples_anchor_any = samples_neighbor_any = samples_combined_any = 0
valid_samples = 0

for i in range(len(data_list)):
    s_key = f"normalized_shapley_values_{seed}"
    if s_key not in data_dict[i]:
        continue

    shapley = np.array(data_dict[i][s_key])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]
    fragment_global_idx = np.where(fragment_mask)[0]

    # Robust anchor extraction (mask-style or index-style)
    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    # Keep only fragment anchors
    frag_set = set(fragment_global_idx.tolist())
    anchor_global_idx = np.array([a for a in anchor_global_idx if a in frag_set], dtype=int)

    # Top-k in shapley index space
    k = min(top_k, shapley.size)
    topk_idx = np.argsort(shapley)[::-1][:k]

    # Map global atom index -> shapley index space
    if shapley.size == fragment_global_idx.size:
        global_to_shapley = {g: l for l, g in enumerate(fragment_global_idx)}
    else:
        global_to_shapley = {g: g for g in range(min(n_atoms, shapley.size))}

    anchor_idx = np.array([global_to_shapley[g] for g in anchor_global_idx if g in global_to_shapley], dtype=int)

    # Neighbors: fragment atoms with distance to linker COM <= threshold
    d_key = f"distances_{seed}"
    if d_key in data_dict[i]:
        d_frag = np.array(data_dict[i][d_key])
    else:
        pos_key = f"positions_{seed}"
        com_key = f"com_linker_{seed}"
        if pos_key not in data_dict[i] or com_key not in data_dict[i]:
            continue
        frag_pos = np.array(data_dict[i][pos_key])[fragment_mask]
        com_linker = np.array(data_dict[i][com_key])
        d_frag = np.linalg.norm(frag_pos - com_linker, axis=1)

    if d_frag.size != fragment_global_idx.size:
        continue

    neighbor_global_idx = fragment_global_idx[d_frag <= distance_threshold]
    # Exclude anchors from neighbor-only set
    neighbor_global_idx = np.array([g for g in neighbor_global_idx if g not in set(anchor_global_idx.tolist())], dtype=int)

    neighbor_idx = np.array([global_to_shapley[g] for g in neighbor_global_idx if g in global_to_shapley], dtype=int)
    combined_idx = np.unique(np.concatenate([anchor_idx, neighbor_idx])) if (anchor_idx.size + neighbor_idx.size) > 0 else np.array([], dtype=int)

    anchor_hits = np.intersect1d(topk_idx, anchor_idx).size
    neighbor_hits = np.intersect1d(topk_idx, neighbor_idx).size
    combined_hits = np.intersect1d(topk_idx, combined_idx).size

    tot_anchor_hits += anchor_hits
    tot_neighbor_hits += neighbor_hits
    tot_combined_hits += combined_hits
    tot_topk_slots += k

    tot_anchor_atoms += anchor_idx.size
    tot_neighbor_atoms += neighbor_idx.size
    tot_combined_atoms += combined_idx.size

    samples_anchor_any += int(anchor_hits > 0)
    samples_neighbor_any += int(neighbor_hits > 0)
    samples_combined_any += int(combined_hits > 0)
    valid_samples += 1

anchor_com_neighbor_topk_stats[seed] = {
    "anchor_hit_rate_in_topk_slots": (tot_anchor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "neighbor_hit_rate_in_topk_slots": (tot_neighbor_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_or_neighbor_hit_rate_in_topk_slots": (tot_combined_hits / tot_topk_slots) if tot_topk_slots > 0 else np.nan,
    "anchor_coverage_rate": (tot_anchor_hits / tot_anchor_atoms) if tot_anchor_atoms > 0 else np.nan,
    "neighbor_coverage_rate": (tot_neighbor_hits / tot_neighbor_atoms) if tot_neighbor_atoms > 0 else np.nan,
    "anchor_or_neighbor_coverage_rate": (tot_combined_hits / tot_combined_atoms) if tot_combined_atoms > 0 else np.nan,
    "sample_freq_any_anchor_in_topk": (samples_anchor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_neighbor_in_topk": (samples_neighbor_any / valid_samples) if valid_samples > 0 else np.nan,
    "sample_freq_any_anchor_or_neighbor_in_topk": (samples_combined_any / valid_samples) if valid_samples > 0 else np.nan,
    "valid_samples": valid_samples,
    "top_k": top_k,
    "distance_threshold_angstrom": distance_threshold,
}

stats_com = anchor_com_neighbor_topk_stats[seed]
print(f"Seed {seed} | top_k={top_k}, neighbor distance <= {distance_threshold:.1f} Å from linker COM")
for k, v in stats_com.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Seed 42 | top_k=5, neighbor distance <= 2.0 Å from linker COM
  anchor_hit_rate_in_topk_slots: 0.2200
  neighbor_hit_rate_in_topk_slots: 0.0133
  anchor_or_neighbor_hit_rate_in_topk_slots: 0.2333
  anchor_coverage_rate: 0.5500
  neighbor_coverage_rate: 1.0000
  anchor_or_neighbor_coverage_rate: 0.5645
  sample_freq_any_anchor_in_topk: 0.7667
  sample_freq_any_neighbor_in_topk: 0.0667
  sample_freq_any_anchor_or_neighbor_in_topk: 0.7667
  valid_samples: 30
  top_k: 5
  distance_threshold_angstrom: 2.0000


Monte Carlo permutation (randomization) test with a one-sided alternative to build a null distribution - test against randomicity of importance ranking

In [28]:
# Permutation test (ANCHORS ONLY): observed top-k vs random top-k
n_perm = 20000
rng = np.random.default_rng(42)

obs_anchor_hits = 0
obs_topk_slots = 0
obs_samples_any_anchor = 0
valid_samples = 0

# Cache per-sample index sets once
cache = []

for i in range(len(data_list)):
    s_key = f"normalized_shapley_values_{seed}"
    if s_key not in data_dict[i]:
        continue
    shapley = np.array(data_dict[i][s_key])
    if shapley.size == 0:
        continue

    fragment_mask = data_dict[i]["fragment_mask"].bool().cpu().numpy().squeeze()
    n_atoms = fragment_mask.shape[0]
    fragment_global_idx = np.where(fragment_mask)[0]

    anchors_raw = data_dict[i]["anchors"].cpu().numpy().squeeze()
    anchors_flat = np.array(anchors_raw).reshape(-1)

    # anchor extraction
    if anchors_flat.size == n_atoms and np.all(np.isin(anchors_flat, [0, 1])):
        anchor_global_idx = np.where(anchors_flat > 0)[0]
    else:
        anchor_global_idx = anchors_flat.astype(int)
        anchor_global_idx = anchor_global_idx[(anchor_global_idx >= 0) & (anchor_global_idx < n_atoms)]
        anchor_global_idx = np.unique(anchor_global_idx)

    # keep only anchors inside fragment atoms
    frag_set = set(fragment_global_idx.tolist())
    anchor_global_idx = np.array([a for a in anchor_global_idx if a in frag_set], dtype=int)

    k = min(top_k, shapley.size)
    topk_obs = np.argsort(shapley)[::-1][:k]

    # map global atom index -> shapley index space
    if shapley.size == fragment_global_idx.size:
        global_to_shapley = {g: l for l, g in enumerate(fragment_global_idx)}
    else:
        global_to_shapley = {g: g for g in range(min(n_atoms, shapley.size))}

    anchor_idx = np.array([global_to_shapley[g] for g in anchor_global_idx if g in global_to_shapley], dtype=int)

    # observed (anchors only)
    anchor_hits = np.intersect1d(topk_obs, anchor_idx).size
    obs_anchor_hits += anchor_hits
    obs_topk_slots += k
    obs_samples_any_anchor += int(anchor_hits > 0)
    valid_samples += 1

    cache.append((shapley.size, k, anchor_idx))

obs_hit_rate = obs_anchor_hits / obs_topk_slots if obs_topk_slots > 0 else np.nan
obs_sample_freq = obs_samples_any_anchor / valid_samples if valid_samples > 0 else np.nan

# null distribution
null_hit_rates = np.zeros(n_perm, dtype=float)
null_sample_freqs = np.zeros(n_perm, dtype=float)

for p in range(n_perm):
    rnd_hits = 0
    rnd_slots = 0
    rnd_any = 0
    for N, k, anchor_idx in cache:
        rnd_topk = rng.choice(N, size=k, replace=False)
        h = np.intersect1d(rnd_topk, anchor_idx).size
        rnd_hits += h
        rnd_slots += k
        rnd_any += int(h > 0)
    null_hit_rates[p] = rnd_hits / rnd_slots if rnd_slots > 0 else np.nan
    null_sample_freqs[p] = rnd_any / len(cache) if len(cache) > 0 else np.nan

# one-sided empirical p-values: observed > random
p_hit = (1 + np.sum(null_hit_rates >= obs_hit_rate)) / (n_perm + 1)
p_freq = (1 + np.sum(null_sample_freqs >= obs_sample_freq)) / (n_perm + 1)

print(f"Observed anchor_hit_rate_in_topk_slots: {obs_hit_rate:.4f}")
print(f"Random mean±std: {np.nanmean(null_hit_rates):.4f} ± {np.nanstd(null_hit_rates, ddof=1):.4f}")
print(f"Empirical p-value (hit rate): {p_hit:.4g}")

print(f"Observed sample_freq_any_anchor_in_topk: {obs_sample_freq:.4f}")
print(f"Random mean±std: {np.nanmean(null_sample_freqs):.4f} ± {np.nanstd(null_sample_freqs, ddof=1):.4f}")
print(f"Empirical p-value (sample freq): {p_freq:.4g}")

# Effect sizes
z_hit = (obs_hit_rate - np.nanmean(null_hit_rates)) / (np.nanstd(null_hit_rates, ddof=1) + 1e-12)
z_freq = (obs_sample_freq - np.nanmean(null_sample_freqs)) / (np.nanstd(null_sample_freqs, ddof=1) + 1e-12)

# 95% null intervals
ci_hit = np.nanpercentile(null_hit_rates, [2.5, 97.5])
ci_freq = np.nanpercentile(null_sample_freqs, [2.5, 97.5])

print(f"Z-score (hit rate): {z_hit:.2f}")
print(f"Z-score (sample freq): {z_freq:.2f}")
print(f"Null 95% interval (hit rate): [{ci_hit[0]:.4f}, {ci_hit[1]:.4f}]")
print(f"Null 95% interval (sample freq): [{ci_freq[0]:.4f}, {ci_freq[1]:.4f}]")

print(f"Significant hit-rate enrichment? {'YES' if p_hit < 0.05 else 'NO'}")
print(f"Significant sample-freq enrichment? {'YES' if p_freq < 0.05 else 'NO'}")

Observed anchor_hit_rate_in_topk_slots: 0.2200
Random mean±std: 0.1002 ± 0.0216
Empirical p-value (hit rate): 5e-05
Observed sample_freq_any_anchor_in_topk: 0.7667
Random mean±std: 0.4469 ± 0.0902
Empirical p-value (sample freq): 0.00025
Z-score (hit rate): 5.54
Z-score (sample freq): 3.54
Null 95% interval (hit rate): [0.0600, 0.1467]
Null 95% interval (sample freq): [0.2667, 0.6333]
Significant hit-rate enrichment? YES
Significant sample-freq enrichment? YES


In [30]:
null_hit_mean = np.nanmean(null_hit_rates)
null_freq_mean = np.nanmean(null_sample_freqs)

p_hit_2s = (
    1
    + np.sum(np.abs(null_hit_rates - null_hit_mean) >= np.abs(obs_hit_rate - null_hit_mean))
) / (n_perm + 1)

p_freq_2s = (
    1
    + np.sum(np.abs(null_sample_freqs - null_freq_mean) >= np.abs(obs_sample_freq - null_freq_mean))
) / (n_perm + 1)

print(f"Empirical two-sided p-value (hit rate): {p_hit_2s:.4g}")
print(f"Empirical two-sided p-value (sample freq): {p_freq_2s:.4g}")

print(f"Significant hit-rate deviation (two-sided)? {'YES' if p_hit_2s < 0.05 else 'NO'}")
print(f"Significant sample-freq deviation (two-sided)? {'YES' if p_freq_2s < 0.05 else 'NO'}")

Empirical two-sided p-value (hit rate): 5e-05
Empirical two-sided p-value (sample freq): 0.00025
Significant hit-rate deviation (two-sided)? YES
Significant sample-freq deviation (two-sided)? YES
